In [ ]:
# Example 1: Parse a Newick tree string and inspect structure.
import dendropy

newick_str = "((A:1.0, B:2.0):1.5, (C:3.0, D:4.0):0.5);"
tree = dendropy.Tree.get(data=newick_str, schema="newick")

leaf_labels = sorted(leaf.taxon.label for leaf in tree.leaf_node_iter())
print(f"Leaf labels: {leaf_labels}")
print(f"Number of leaves: {len(leaf_labels)}")
print(f"Number of internal nodes: {len(list(tree.preorder_internal_node_iter()))}")

assert leaf_labels == ["A", "B", "C", "D"]
assert len(leaf_labels) == 4


In [ ]:
# Example 2: Build a TaxonNamespace and a DnaCharacterMatrix in-memory.
import dendropy

taxa = dendropy.TaxonNamespace(["s1", "s2", "s3"])
dna = dendropy.DnaCharacterMatrix.from_dict(
    {"s1": "ACGTACGT", "s2": "ACGTACGA", "s3": "ACGTACGC"},
    taxon_namespace=taxa,
)

print(f"Num taxa: {len(dna.taxon_namespace)}")
print(f"Sequence length: {dna.max_sequence_size}")
for taxon, seq in dna.items():
    print(f"  {taxon.label}: {seq.symbols_as_string()}")

assert len(dna.taxon_namespace) == 3
assert dna.max_sequence_size == 8


In [ ]:
# Example 3: Round-trip a tree through Newick (write then read back).
import io

import dendropy

newick_str = "((A:1.0, B:2.0):1.5, (C:3.0, D:4.0):0.5);"
tree = dendropy.Tree.get(data=newick_str, schema="newick")

buf = io.StringIO()
tree.write(file=buf, schema="newick")
serialized = buf.getvalue()
print(f"Serialized: {serialized.strip()}")

tree2 = dendropy.Tree.get(data=serialized, schema="newick")
labels1 = sorted(leaf.taxon.label for leaf in tree.leaf_node_iter())
labels2 = sorted(leaf.taxon.label for leaf in tree2.leaf_node_iter())

assert labels1 == labels2
print(f"Round-trip preserved leaves: {labels1}")


In [ ]:
# Example 4: TreeList and basic tree statistics.
import dendropy

trees_str = "((A:1.0, B:2.0):1.5, (C:3.0, D:4.0):0.5);\n((A:1.0, C:2.0):1.5, (B:3.0, D:4.0):0.5);"
tree_list = dendropy.TreeList.get(data=trees_str, schema="newick")

print(f"Number of trees: {len(tree_list)}")
for i, tree in enumerate(tree_list):
    total_length = tree.length()
    print(f"  Tree {i}: total branch length = {total_length}")

assert len(tree_list) == 2
assert all(t.length() > 0 for t in tree_list)


In [ ]:
# Example 5: Compute symmetric (Robinson-Foulds) distance between two trees.
import dendropy
from dendropy.calculate import treecompare

taxa = dendropy.TaxonNamespace()
t1 = dendropy.Tree.get(
    data="((A,B),(C,D));", schema="newick", taxon_namespace=taxa
)
t2 = dendropy.Tree.get(
    data="((A,C),(B,D));", schema="newick", taxon_namespace=taxa
)
t3 = dendropy.Tree.get(
    data="((A,B),(C,D));", schema="newick", taxon_namespace=taxa
)

rf_diff = treecompare.symmetric_difference(t1, t2)
rf_same = treecompare.symmetric_difference(t1, t3)
print(f"RF(t1, t2) = {rf_diff}")
print(f"RF(t1, t3) = {rf_same}")

assert rf_same == 0
assert rf_diff > 0
